# P4 final independent second-life perturbations

This cleaned notebook shows the optimized first life followed by six independent EOFU restart cases. Each non-nominal branch changes exactly one DeepSOH state.

- First use: 100 cycles, 1.50 C charge, 1.25 C discharge, 3.1–4.1 V, C/50 hold, 0.1 h rest.
- Second use: 2.00 C charge, 1.25 C discharge, 3.0–4.1 V, C/50 hold, 0.1 h rest.
- Solver: RK23, `rtol=1e-7`, scaled absolute tolerance, first step 10 cycles.
- Stopping rule: retain each second-life branch through the last cycle at or above 80% of its own starting capacity.

The final plots exclude $\delta_{\mathrm{SEI}}\times0.1$ as requested. That older trajectory remains only in the bundled raw data and audit record.

## Load and validate the final six cases

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# Locate the deliverable folder from the working directory, so the notebook
# runs from a fresh checkout with no machine-specific path.
FINAL_ROOT = Path.cwd()
if not (FINAL_ROOT / "data").exists():
    for _base in (Path.cwd(), *Path.cwd().parents):
        _cand = _base / "deepSOH_final_august22_2026"
        if (_cand / "data").exists():
            FINAL_ROOT = _cand
            break
DATA_ROOT = FINAL_ROOT / "data" / "independent_perturbations"
OUTPUT_ROOT = FINAL_ROOT / "outputs" / "independent_perturbations"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

first = pd.read_csv(DATA_ROOT / "P4_optimized_first_life_trajectory.csv")
second_all = pd.read_csv(DATA_ROOT / "P4_perturbed_second_life_trajectories.csv")
states_all = pd.read_csv(DATA_ROOT / "P4_historical_perturbation_initial_states.csv")
summary = json.loads((DATA_ROOT / "P4_historical_perturbations_summary.json").read_text())
sei_audit = json.loads((DATA_ROOT / "P4_delta_SEI_audit.json").read_text())

FINAL_CASES = (
    "nominal", "nLi_x0p95", "Cp_x0p85", "Cn_x0p90",
    "plating_x1p5", "SEI_x2",
)
order = {case: index for index, case in enumerate(FINAL_CASES)}
states = states_all.loc[states_all["case"].isin(FINAL_CASES)].copy()
states["plot_order"] = states["case"].map(order)
states = states.sort_values("plot_order")
second = second_all.loc[second_all["case"].isin(FINAL_CASES)].copy()

if set(states["case"]) != set(FINAL_CASES):
    raise AssertionError("One or more final perturbation cases are missing")
if not (states["changed_state_count"] == [0, 1, 1, 1, 1, 1]).all():
    raise AssertionError("Independent-state perturbation validation failed")

display(states[[
    "label", "perturbed_state", "multiplier", "changed_state_count",
    "initial_capacity_Ah", "initial_resistance_Ohm",
    "safe_last_cycle", "safe_final_retention", "maximum_scaled_initial_error",
]].reset_index(drop=True))

## Perturbed second life (second use only)

In [ ]:
def plot_outputs(output_path):
    panels = (("capacity_Ah", "Capacity [Ah]"), ("resistance_Ohm", "Resistance [Ohm]"))
    fig, axes = plt.subplots(2, 1, figsize=(11, 8.5), sharex=True, constrained_layout=True)
    for _, state in states.iterrows():
        block = second.loc[second["case"] == state["case"]].sort_values("cycle")
        for index, (column, ylabel) in enumerate(panels):
            axes[index].plot(
                block["cycle"], block[column], color=state["color"],
                linestyle=state["linestyle"], linewidth=1.8,
                label=state["label"] if index == 0 else None,
            )
            axes[index].set_ylabel(ylabel)
            axes[index].grid(True, alpha=0.28)
    axes[0].legend(loc="upper center", bbox_to_anchor=(0.5, 1.34), ncol=3, fontsize=8.4)
    axes[-1].set_xlabel("Cycle number in second life")
    fig.suptitle("P4 independent perturbations: second life only", fontsize=13)
    fig.savefig(output_path, dpi=220, bbox_inches="tight", facecolor="white")
    plt.show()

plot_outputs(OUTPUT_ROOT / "P4_independent_perturbations_second_life.png")

## DeepSOH state trajectories

In [ ]:
panels = (
    ("nLi_mol", r"$n_{\mathrm{Li}}$ [mol]"),
    ("Cp_Ah", r"$C_p$ [Ah]"),
    ("Cn_Ah", r"$C_n$ [Ah]"),
    ("delta_SEI_m", r"$\delta_{\mathrm{SEI}}$ [m]"),
    ("delta_pl_m", r"$\delta_{\mathrm{plating}}$ [m]"),
)
fig, axes = plt.subplots(3, 2, figsize=(12, 12), sharex=True, constrained_layout=True)
flat = axes.ravel()
for _, state in states.iterrows():
    block = second.loc[second["case"] == state["case"]].sort_values("cycle")
    for index, (column, ylabel) in enumerate(panels):
        flat[index].plot(
            block["cycle"], block[column], color=state["color"],
            linestyle=state["linestyle"], linewidth=1.6,
            label=state["label"] if index == 0 else None,
        )
        flat[index].set_ylabel(ylabel)
        flat[index].set_xlabel("Second-life cycle")
        flat[index].grid(True, alpha=0.28)
flat[0].legend(loc="best", fontsize=7.2)
flat[5].axis("off")
fig.suptitle("P4 DeepSOH states after independent EOFU perturbations", fontsize=13)
state_png = OUTPUT_ROOT / "P4_independent_perturbation_states.png"
fig.savefig(state_png, dpi=220, bbox_inches="tight", facecolor="white")
plt.show()

## SEI restart note

The restart assigns the target total SEI thickness to the initial outer-SEI thickness and sets the inner thickness to zero. The active P4 model is EC-reaction-limited, so thickness enters an additive `1 + ...` denominator rather than a pure $1/L_{\mathrm{SEI}}$ law. This explains why the future-capacity effect of changing initial $\delta_{\mathrm{SEI}}$ is modest.

In [ ]:
display(pd.DataFrame(sei_audit["trajectory_comparisons_to_nominal"]))

## Export plotting data to MATLAB (.mat)

In [ ]:
from scipy.io import savemat

MAT_ROOT = FINAL_ROOT / "matlab"
MAT_ROOT.mkdir(parents=True, exist_ok=True)

def _hex2rgb(h):
    h = h.lstrip("#")
    return [int(h[i:i+2], 16) / 255.0 for i in (0, 2, 4)]

def _mls(ls):
    return ls if isinstance(ls, str) and ls in ("-", "--", "-.", ":") else "-"

_cols = ["cycle", "capacity_Ah", "resistance_Ohm",
         "nLi_mol", "Cp_Ah", "Cn_Ah", "delta_SEI_m", "delta_pl_m"]
mat = {}
_cases, _colors, _ls = [], [], []
for _, st in states.iterrows():
    case = st["case"]
    _cases.append(case)
    block = second[second["case"] == case].sort_values("cycle")
    mat[case] = {c: block[c].to_numpy(dtype=float) for c in _cols}
    _colors.append(_hex2rgb(st["color"]))
    _ls.append(_mls(st["linestyle"]))
mat["cases"] = np.array(_cases, dtype=object)
mat["colors"] = np.array(_colors, dtype=float)
mat["linestyles"] = np.array(_ls, dtype=object)
mat["state_columns"] = np.array(["nLi_mol", "Cp_Ah", "Cn_Ah", "delta_SEI_m", "delta_pl_m"], dtype=object)
savemat(str(MAT_ROOT / "P4_02_independent_perturbations.mat"), mat, do_compression=True)
print("saved", MAT_ROOT / "P4_02_independent_perturbations.mat")